# Group 11 — Milestone 2: parse full 5 khoảng của nhóm (Colab CPU-only)

Notebook này parse **toàn bộ 25.000 ID của nhóm** (5 khoảng × 5000 ID), lần lượt trên cùng một runtime Colab CPU-only. Cùng một lần chạy cho ra cả **3 bảng thật** (`elements`, `hierarchy`, `bibtex`) dùng cho M3 lẫn **số liệu benchmark M2** cho book.

**Trước khi chạy (làm một lần):**
1. *Runtime → Change runtime type → **CPU***. Đề yêu cầu benchmark trên Colab CPU-only.
2. Thư mục **`benchmark_ms1_main`** (dữ liệu M1 cả nhóm) đang nằm ở *Được chia sẻ với tôi*. Bấm chuột phải vào nó, chọn *Sắp xếp → Thêm lối tắt* (*Add shortcut to Drive*), đặt lối tắt ngay trong *Drive của tôi*. Không cần tải về hay copy.
3. Chạy hết từ trên xuống (*Runtime → Run all*).

**Thời gian ước tính:** khôi phục dữ liệu khoảng 10–20 phút mỗi khoảng; parse khoảng 25 phút mỗi khoảng. Tổng khoảng 3–4 giờ, dưới giới hạn 12 giờ của Colab.

**Cái gì được đo:** chỉ `benchmark_m2.py` của từng khoảng (mục 8) và bước gộp bảng `--export-only` (mục 9). Cài đặt, khôi phục dữ liệu, test, validate, thống kê và lưu Drive **không** tính vào benchmark.

**Nếu Colab bị ngắt:** mỗi khoảng chạy xong được lưu lên Drive (`g11_m2_benchmark/<tên>_m2.tar.gz`). Mở runtime mới rồi *Run all* lại: khoảng nào đã xong thì được khôi phục và bỏ qua. Khoảng đang chạy dở sẽ chạy lại từ đầu, không lấy số liệu dở dang. Các khoảng chạy ở runtime khác nhau có thể có CPU khác nhau; mục 10 in CPU của từng khoảng để ghi vào book.

**Code:** lấy đúng tag **`m2-final`** trên GitHub (code M2 đã chốt), không lấy nhánh đang thay đổi. Nhờ vậy cả 5 khoảng chắc chắn chạy cùng một code.

## 1. Xác minh môi trường CPU và thông tin phiên chạy

In [ ]:
import os, sys, re, shutil, platform, subprocess, json, csv, tarfile, datetime
from pathlib import Path
import psutil

CONTENT = Path("/content")

def cpu_model():
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

print("UTC time:", utc_now())
print("Python:", sys.version)
print("CPU model:", cpu_model())
print("CPU logical:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(psutil.virtual_memory().total / 1e9, 2))
print("Disk free (GB):", round(shutil.disk_usage(CONTENT).free / 1e9, 1))
print("Platform:", platform.platform())
print("COLAB_GPU env:", os.environ.get("COLAB_GPU", "not set"))
print("nvidia-smi path:", shutil.which("nvidia-smi"))
print("MANUAL CHECK REQUIRED: Runtime -> Change runtime type -> CPU")

## 2. Mount Google Drive – nơi chứa dữ liệu M1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Lấy code M2 đã chốt (tag `m2-final`) và cài thư viện

In [ ]:
REPO_URL = "https://github.com/mituleefam/group11-arxiv-project.git"
CODE_TAG = "m2-final"   # tag của code M2 đã chốt. KHÔNG đổi, trừ khi cả nhóm thống nhất chốt lại code
REPO_DIR = CONTENT / "group11-arxiv-project"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", "--branch", CODE_TAG, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
actual_tag = subprocess.run(["git", "-C", str(REPO_DIR), "describe", "--tags", "--exact-match"],
                            capture_output=True, text=True).stdout.strip()
assert actual_tag == CODE_TAG, (f"Code trong {REPO_DIR} là {actual_tag or 'không phải tag nào'}, cần {CODE_TAG}. "
                                "Mở runtime mới (Runtime → Disconnect and delete runtime) rồi chạy lại.")
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Code tag:", actual_tag)
print("Commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "src" / "requirements.txt")], check=True)
import pyarrow
print("pyarrow:", pyarrow.__version__)

## 4. Cấu hình — phần duy nhất người chạy có thể cần sửa

Dữ liệu M1 trên Drive có dạng sau (mỗi khoảng một thư mục, do `chunked_crawl.py` của M1 lưu):

```
benchmark_ms1_main/                       ← lối tắt trong "Drive của tôi"
├── Tram_2511-21709_2512-03230/
│   ├── snapshots/
│   │   ├── 0001_discovery.tar.gz         ← các bản lưu TĂNG DẦN của một lần crawl:
│   │   ├── 0002_chunk-....tar.gz            mỗi file chỉ chứa phần thay đổi so với file trước
│   │   └── ... 0009_late-sweep-1.tar.gz     (kể cả danh sách file đã xóa)
│   ├── perf/, validation_report.csv, ... ← log M1, M2 không dùng
├── Thu_2512-13231_2512-18230/
└── ...
```

Vì các file trong `snapshots/` là bản lưu tăng dần, **không được giải nén từng file rồi chép đè**: làm vậy sẽ giữ lại cả những file M1 đã xóa. Notebook khôi phục bằng `snapshot.py restore` (đúng công cụ M1 đã dùng), áp các file theo thứ tự `0001 → 0009` và xóa đúng những file cần xóa.

**File `.bbl` không cần tách ra.** Crawl M1 bản cuối đã giữ `.bbl` ngay trong thư mục version (`corpus/<id>/vN/`), cạnh `.tex` và `.bib`. Parser M2 đọc chúng tại đó. Mục 6 chỉ đếm để kiểm tra.

Dữ liệu trên Drive chỉ được **đọc**, không bao giờ bị sửa. Mọi thứ được làm trên bản copy trong `/content`.

In [ ]:
DRIVE_ROOT = CONTENT / "drive" / "MyDrive"

# ── (1) Thư mục dữ liệu M1 của cả nhóm ─────────────────────────────────────────
# Đường dẫn tới lối tắt "benchmark_ms1_main" trong Drive của tôi.
# Nếu bạn đặt lối tắt trong một thư mục con, sửa dòng này, ví dụ: DRIVE_ROOT / "IntroDS" / "benchmark_ms1_main"
M1_ROOT = DRIVE_ROOT / "benchmark_ms1_main"

# ── (2) Các khoảng cần chạy: tên ngắn -> tên thư mục của khoảng trong M1_ROOT ──
# Tên ngắn (bên trái) dùng để đặt tên kết quả, chỉ gồm chữ, số, "_" hoặc "-".
# Muốn chạy thử một khoảng: thêm dấu # trước các dòng còn lại.
# Muốn chia cho nhiều người chạy song song: mỗi người giữ một vài dòng, nhưng khi đó
#   bước gộp (mục 9-10) phải chạy ở một runtime có đủ kết quả của cả 5 khoảng.
RANGES = {
    "Tram": "Tram_2511-21709_2512-03230",
    "Thu":  "Thu_2512-13231_2512-18230",
    "Phat": "Phat_2601-13156_2601-18155",
    "Bao":  "Bao_2604-05535_2604-10534",
    "Quan": "Quan_2605-12338_2605-17337",
}

# ── (3) Thông số chạy: GIỮ NGUYÊN để số liệu 5 khoảng so sánh được với nhau ───
EXPECTED_PER_RANGE = 5000   # số ID được giao cho mỗi khoảng (để kiểm tra độ bao phủ)
WORKERS = 2                 # số tiến trình parse song song; Colab CPU có 2 lõi

# ── (4) Tải bù .bbl: để False ──────────────────────────────────────────────────
# Chỉ cần True với dữ liệu crawl TRƯỚC 08/11 (lúc crawler chưa giữ .bbl). Dữ liệu
# benchmark_ms1_main đã có .bbl; bật True sẽ tải lại hàng nghìn bài từ arXiv chỉ
# để thấy chúng không có .bbl, mất nhiều giờ. Mục 6 sẽ cảnh báo nếu cần bật.
RUN_FETCH_BBL = False

# ── (5) Nơi lưu kết quả (trên Drive của NGƯỜI CHẠY, không phải thư mục dùng chung) ──
SAVE_MERGED_TABLES = True                       # lưu 3 bảng Parquet đã gộp (input của M3 và Hugging Face)
RESULT_DRIVE = DRIVE_ROOT / "g11_m2_benchmark"  # kết quả từng khoảng + kết quả gộp
BBL_DRIVE_DIR = DRIVE_ROOT / "m2_bbl"           # chỉ dùng khi RUN_FETCH_BBL = True

# ── Phần dưới không cần sửa ─────────────────────────────────────────────────────
WORK_ROOT = CONTENT / "m2_work"                 # bản copy để parse: m2_work/<tên ngắn>
MERGED = WORK_ROOT / "_merged"
MERGED_TABLES = MERGED / "tables"
EXPORT_LOGS = MERGED / "logs"
GROUP_OUT = MERGED / "group_report"

def snapshot_dir(name):
    return M1_ROOT / RANGES[name] / "snapshots"

def snapshot_archives(name):
    return sorted(p for p in snapshot_dir(name).glob("*.tar.gz") if re.match(r"\d{4,}", p.name))

def work_dir(name):
    return WORK_ROOT / name

def is_done(name):
    return (work_dir(name) / "m2_done.json").exists()

def sh(cmd, check=True):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=REPO_DIR, check=check)

assert RANGES, "RANGES đang rỗng: giữ lại ít nhất một khoảng"
assert M1_ROOT.is_dir(), (f"Không thấy {M1_ROOT}. Đã thêm lối tắt 'benchmark_ms1_main' vào Drive của tôi chưa? "
                          "Nếu lối tắt nằm trong thư mục con thì sửa M1_ROOT ở trên.")
for name, folder in RANGES.items():
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", name), f"Tên ngắn không hợp lệ: {name}"
    archives = snapshot_archives(name)
    assert archives, f"{name}: không có file NNNN_*.tar.gz trong {snapshot_dir(name)}"
    numbers = [int(re.match(r"\d+", a.name).group()) for a in archives]
    gap = sorted(set(range(1, max(numbers) + 1)) - set(numbers))
    assert not gap, f"{name}: thiếu snapshot số {gap} trong {snapshot_dir(name)}; hỏi người crawl khoảng này"
    size = sum(a.stat().st_size for a in archives) / 1e9
    print(f"{name:5s} {folder}: {len(archives)} snapshot, {size:.2f} GB, cuối cùng {archives[-1].name}")
print("Kết quả sẽ lưu ở:", RESULT_DRIVE)

## 5. Khôi phục dữ liệu M1 của từng khoảng vào `/content`

Với mỗi khoảng: `snapshot.py restore` áp các snapshot `0001 → cuối` vào `m2_work/<tên>/`, đúng như lúc M1 khôi phục giữa các bước crawl. Sau đó cell kiểm tra có `corpus/` và `work/paper_manifest.jsonl` (danh sách ID M1 tìm thấy, M2 dùng nó để biết bài nào cần parse).

Nếu khoảng đó đã chạy xong ở một runtime trước, kết quả (`parsed/`, `logs/`) được lấy lại từ Drive và mục 8 sẽ bỏ qua khoảng đó.

In [ ]:
def load_range(name):
    d = work_dir(name)
    if (d / ".loaded").exists():
        print(f"{name}: đã có trong {d}")
        return
    if d.exists():  # một lần khôi phục trước bị ngắt giữa chừng: làm lại từ đầu
        print(f"{name}: xóa {d} còn dở từ lần trước")
        shutil.rmtree(d)
    print(f"{name}: khôi phục {len(snapshot_archives(name))} snapshot từ {snapshot_dir(name)}", flush=True)
    sh([sys.executable, "src/snapshot.py", "restore", "--data-dir", d, "--backup-dir", snapshot_dir(name)])
    assert (d / "corpus").is_dir(), f"{name}: sau khi khôi phục không có {d / 'corpus'}"
    for parsed in (d / "corpus").glob("*/parsed"):  # kết quả parse cũ (nếu có) không được dùng lại
        shutil.rmtree(parsed)
    manifest = d / "work" / "paper_manifest.jsonl"
    if not manifest.exists():
        print(f"CẢNH BÁO {name}: không có work/paper_manifest.jsonl, M2 sẽ parse mọi thư mục trong corpus/")
    papers = sum(p.is_dir() for p in (d / "corpus").iterdir())
    assert papers > 0, f"{name}: corpus/ không có bài nào"
    (d / ".loaded").write_text(json.dumps({"snapshots": [a.name for a in snapshot_archives(name)],
                                           "loaded_utc": utc_now(), "paper_folders": papers}), encoding="utf-8")
    print(f"{name}: {papers} thư mục bài trong {d}")

def restore_range(name):
    archive = RESULT_DRIVE / f"{name}_m2.tar.gz"
    if is_done(name) or not archive.exists():
        return
    with tarfile.open(archive, mode="r:gz") as tar:
        tar.extractall(work_dir(name), filter="data")
    print(f"{name}: lấy lại kết quả đã chạy xong từ {archive}; mục 8 sẽ bỏ qua khoảng này")

for name in RANGES:
    load_range(name)
    restore_range(name)
    print("  Disk còn trống (GB):", round(shutil.disk_usage(CONTENT).free / 1e9, 1))
print("Đã xong:", [n for n in RANGES if is_done(n)], "| Cần chạy:", [n for n in RANGES if not is_done(n)])

## 6. Kiểm tra file `.bbl` (không tải gì, trừ khi bật `RUN_FETCH_BBL`)

Một số bài chỉ có danh mục tham khảo trong file `.bbl` (không có `.bib`). Crawl M1 bản cuối đã giữ các file này trong thư mục version, nên **không cần tách hay di chuyển `.bbl`**.

Cell này đếm số file `.bbl` của mỗi khoảng và chạy `fetch_bbl.py --dry-run` (chỉ đếm, không gửi request) để biết còn bao nhiêu version có trích dẫn mà không có nguồn tham khảo nào. Với dữ liệu đã giữ `.bbl`, các version này thường là bài mà chính source trên arXiv không có `.bbl`, tải lại cũng không thêm được gì.

Nếu một khoảng có **0 file `.bbl`**, nhiều khả năng khoảng đó được crawl bằng code cũ: báo người crawl, hoặc đặt `RUN_FETCH_BBL = True` ở mục 4 rồi chạy lại từ mục 4. Khi bật, các `.bbl` tải về được lưu vào `m2_bbl/<tên>.tar.gz` trên Drive theo từng đợt, để runtime sau không phải tải lại. Bước tải cần mạng và **không** tính vào benchmark.

In [ ]:
def bbl_archive(name):
    return BBL_DRIVE_DIR / f"{name}.tar.gz"

def save_bbl(name):
    """Gom các .bbl đã tải, versions.json của bài có .bbl và log của một khoảng thành một file nén trên Drive."""
    d = work_dir(name)
    temporary = CONTENT / "m2_bbl.tar.gz"
    with tarfile.open(temporary, "w:gz") as archive:
        papers = set()
        for bbl in (d / "corpus").rglob("*.bbl"):
            archive.add(bbl, arcname=str(bbl.relative_to(d)))
            papers.add(bbl.relative_to(d / "corpus").parts[0])
        for paper in sorted(papers):
            archive.add(d / "corpus" / paper / "versions.json", arcname=f"corpus/{paper}/versions.json")
        log = d / "logs" / "bbl_log.csv"
        if log.exists():
            archive.add(log, arcname="logs/bbl_log.csv")
    BBL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(temporary, bbl_archive(name))
    temporary.unlink()

def fetch_bbl(name):
    d, saved = work_dir(name), bbl_archive(name)
    if saved.exists():
        with tarfile.open(saved) as archive:
            archive.extractall(d, filter="data")
        print(f"{name}: lấy lại các .bbl đã tải từ {saved}")
    fetch = [sys.executable, "src/fetch_bbl.py", "--data-dir", d]
    while True:  # từng đợt 300 bài, lưu lên Drive sau mỗi đợt để Colab có ngắt cũng không mất
        cmd = [str(c) for c in fetch + ["--limit", "300"]]
        run = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        print("\n".join(run.stdout.splitlines()[-2:]), run.stderr[-500:])
        assert run.returncode in (0, 1), (f"{name}: fetch_bbl.py dừng với mã {run.returncode}, chạy lại cell này "
                                          "(mã 2: đợi khóa crawler.lock hết hạn, tối đa 3 phút)")
        save_bbl(name)
        if not re.search(r"(?:added|no_bbl|not_found)=", run.stdout):
            break  # không còn gì để tải, hoặc chỉ còn version lỗi: chạy lại cell này để thử lại chúng

for name in RANGES:
    if is_done(name):
        print(f"{name}: đã chạy xong trước đó, bỏ qua")
        continue
    d = work_dir(name)
    count = sum(1 for _ in (d / "corpus").rglob("*.bbl"))
    print(f"===== {name}: {count} file .bbl trong corpus")
    if count == 0:
        print(f"CẢNH BÁO {name}: không có file .bbl nào. Có thể khoảng này crawl bằng code cũ: "
              "báo người crawl, hoặc đặt RUN_FETCH_BBL = True ở mục 4.")
    sh([sys.executable, "src/fetch_bbl.py", "--data-dir", d, "--dry-run"])
    if RUN_FETCH_BBL:
        fetch_bbl(name)

## 7. Chạy unit test, rồi chạy thử 5 bài để kiểm tra contract

In [ ]:
sh([sys.executable, "src/test_m2_bbl.py"])
sh([sys.executable, "src/test_m2_member2.py"])
sh([sys.executable, "-m", "unittest", "discover", "-s", "src", "-p", "test_m2_member4.py", "-v"])
sh([sys.executable, "src/test_m2_member1.py"])
sh([sys.executable, "src/reference_parser/test_m2_member3.py"])
sh([sys.executable, "src/test_m2_full_fixes.py"])
sh([sys.executable, "src/test_m2_case_diagnostics.py"])
sh([sys.executable, "src/test_m2_warning_audit.py"])

todo = [n for n in RANGES if not is_done(n)]
if todo:
    trial = work_dir(todo[0])
    sh([sys.executable, "src/pipeline_m2.py", "--data-dir", trial, "--limit", "5", "--workers", WORKERS, "--force"])
    sh([sys.executable, "src/validate_m2.py", "--data-dir", trial, "--limit", "5", "--require-parquet"])
    print("Check the output: do NOT accept a full-group benchmark if Member 2/3 stand-ins remain.")

## 8. Benchmark chính thức – parse full từng khoảng, lần lượt

Với mỗi khoảng chưa xong, cell này làm các bước sau:
1. Xóa `parsed/`, `tables/` và `logs/` **trong bản copy `m2_work/<tên>`**. Không đụng vào `.tex`, `.bib`, `.bbl`, cũng không đụng vào Drive.
2. Chạy `benchmark_m2.py --force`. **Chỉ bước này được đo.**
3. Chạy validate, `reference_stats` và `statistics_m2`.
4. Ghi `m2_done.json` (commit, CPU, wall time, RAM) rồi lưu `parsed/` và `logs/` của khoảng lên Drive.

Nếu một bước báo lỗi, ví dụ validate có FAIL, cell dừng ở khoảng đó và khoảng đó không được đánh dấu là xong. Sửa lỗi rồi chạy lại cell: các khoảng đã xong được giữ nguyên.

In [ ]:
def save_range(name):
    d = work_dir(name)
    temporary = CONTENT / f"{name}_m2.tar.gz"
    with tarfile.open(temporary, "w:gz") as archive:
        for parsed in sorted((d / "corpus").glob("*/parsed")):
            archive.add(parsed, arcname=str(parsed.relative_to(d)))
        archive.add(d / "logs", arcname="logs")
        if (d / "tables" / "tables.json").exists():
            archive.add(d / "tables" / "tables.json", arcname="tables/tables.json")
        archive.add(d / "m2_done.json", arcname="m2_done.json")  # last: its presence means complete
    RESULT_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(temporary, RESULT_DRIVE / temporary.name)
    temporary.unlink()
    print(f"{name}: saved to {RESULT_DRIVE / temporary.name}")

for name in RANGES:
    d = work_dir(name)
    if is_done(name):
        print(f"{name}: finished earlier, skip")
        continue
    print(f"===== {name}", flush=True)
    assert d.parent == WORK_ROOT and (d / "corpus").is_dir()
    for folder in (d / "corpus").iterdir():
        if folder.is_dir():
            shutil.rmtree(folder / "parsed", ignore_errors=True)
    shutil.rmtree(d / "tables", ignore_errors=True)
    shutil.rmtree(d / "logs", ignore_errors=True)
    sh([sys.executable, "src/benchmark_m2.py", "--data-dir", d, "--workers", WORKERS, "--force",
        "--interval", "0.5", "--disk-interval", "30",
        "--environment", "google_colab", "--cpu-only-declared"])
    sh([sys.executable, "src/validate_m2.py", "--data-dir", d, "--require-parquet"])
    sh([sys.executable, "src/reference_parser/reference_stats.py", "--data-dir", d])
    sh([sys.executable, "src/statistics_m2.py", "--data-dir", d])
    with (d / "logs" / "performance_log_m2.csv").open(encoding="utf-8-sig", newline="") as f:
        run = list(csv.DictReader(f))[-1]
    assert run["status"] == "finished" and run["exit_code"] == "0", run
    done = {"range": name, "finished_utc": utc_now(), "commit": COMMIT, "cpu_model": cpu_model(),
            "cpu_logical": psutil.cpu_count(logical=True), "workers": WORKERS, "run_id": run["run_id"],
            "wall_seconds": float(run["wall_seconds"]),
            "ram_sampled_peak_bytes": float(run["ram_sampled_peak_bytes"]),
            "parquet_final_bytes": float(run["parquet_final_bytes"])}
    (d / "m2_done.json").write_text(json.dumps(done, indent=2), encoding="utf-8")
    save_range(name)
    print(f"{name}: wall {done['wall_seconds'] / 60:.1f} min, peak RAM {done['ram_sampled_peak_bytes'] / 1e9:.2f} GB")

## 9. Gộp bảng của cả nhóm (`--export-only`, có đo)

`pipeline_m2.py --export-only` đọc `parsed/` của tất cả các khoảng và ghi ra 3 bảng Parquet chung. Bước này chạy dưới `benchmark.py`. File `performance_summary_*.json` mà nó tạo ra chính là `--export-summary` mà mục 10 cần. Sau khi gộp, cell kiểm tra số dòng của bảng gộp có đúng bằng tổng số dòng của các khoảng hay không.

In [ ]:
pending = [n for n in RANGES if not is_done(n)]
assert not pending, f"Run mục 8 first; not finished: {pending}"
shutil.rmtree(MERGED_TABLES, ignore_errors=True)
shutil.rmtree(EXPORT_LOGS, ignore_errors=True)
EXPORT_LOGS.mkdir(parents=True)
sh([sys.executable, "src/benchmark.py", "--data-dir", MERGED_TABLES, "--output-dir", EXPORT_LOGS,
    "--interval", "0.5", "--disk-interval", "30", "--",
    sys.executable, "src/pipeline_m2.py", "--data-dir", *[work_dir(n) for n in RANGES],
    "--export-only", "--out", MERGED_TABLES])
summaries = sorted(EXPORT_LOGS.glob("performance_summary_*.json"))
assert len(summaries) == 1, summaries
EXPORT_SUMMARY = summaries[0]

merged = json.loads((MERGED_TABLES / "tables.json").read_text(encoding="utf-8"))
parts = [json.loads((work_dir(n) / "tables" / "tables.json").read_text(encoding="utf-8")) for n in RANGES]
for key in ("papers", "elements_rows", "hierarchy_rows", "bibtex_rows"):
    total = sum(p.get(key, 0) for p in parts)
    assert merged.get(key) == total, f"{key}: merged={merged.get(key)}, sum of ranges={total}"
    print(f"{key}: {merged[key]:,}")
print("Export summary:", EXPORT_SUMMARY)

## 10. Số liệu benchmark của cả nhóm

Cell này chạy `group_benchmark.py --m2` với các quy tắc sau:
- **wall time** = tổng wall time của các khoảng + thời gian bước gộp;
- **RAM đỉnh** = giá trị lớn nhất, có tính cả bước gộp;
- **dung lượng** = kích thước các bảng đã gộp;
- **tỉ lệ dedup** tính lại từ tổng số trước và sau dedup.

Mỗi lần chạy của một khoảng đã tự export bảng riêng của khoảng đó, nên tổng wall time hơi cao hơn một lần chạy liền. Nếu có `WARNING`, đọc kỹ trước khi đưa số liệu vào book. Một khoảng có ít bài hơn `EXPECTED_PER_RANGE`, vì M1 không tìm thấy một số ID, cũng sinh cảnh báo.

In [ ]:
result = sh([sys.executable, "src/group_benchmark.py", "--m2", *[f"{n}={work_dir(n)}" for n in RANGES],
             "--expected-per-member", EXPECTED_PER_RANGE, "--tables-dir", MERGED_TABLES,
             "--export-summary", EXPORT_SUMMARY, "--out", GROUP_OUT], check=False)
assert result.returncode in (0, 2), f"group_benchmark.py failed with code {result.returncode}"
report = json.loads((GROUP_OUT / "group_benchmark_m2.json").read_text(encoding="utf-8"))

def fmt(value, scale, digits):
    return "-" if value is None else f"{value / scale:.{digits}f}"

print(f"{'range':<8}{'papers':>8}{'wall h':>9}{'RAM GB':>9}{'Parquet GB':>12}{'elem dedup %':>14}{'bib dedup %':>13}  CPU")
for row in report["members"] + [report["group"]]:
    name = row["member"]
    cpu = (json.loads((work_dir(name) / "m2_done.json").read_text())["cpu_model"]
           if name in RANGES else "")
    print(f"{name:<8}{fmt(row['papers'], 1, 0):>8}{fmt(row['wall_seconds'], 3600, 2):>9}"
          f"{fmt(row['ram_peak_bytes'], 1e9, 2):>9}{fmt(row['parquet_final_bytes'], 1e9, 3):>12}"
          f"{fmt(row['element_reduction_percent'], 1, 2):>14}{fmt(row['bibtex_reduction_percent'], 1, 2):>13}  {cpu}")
group = report["group"]
if group["wall_seconds"] and group["papers"]:
    print(f"\nAverage per paper: {group['wall_seconds'] / group['papers']:.3f} s")
for warning in report["warnings"]:
    print("WARNING:", warning)
print("\nGROUP:", group["completeness"])

## 11. Lưu kết quả gộp lên Drive (không commit corpus hay bảng Parquet lớn lên GitHub)

In [ ]:
dest = RESULT_DRIVE / "_merged"
dest.mkdir(parents=True, exist_ok=True)
shutil.copytree(GROUP_OUT, dest / "group_report", dirs_exist_ok=True)
shutil.copytree(EXPORT_LOGS, dest / "export_logs", dirs_exist_ok=True)
for name in RANGES:
    shutil.copy2(work_dir(name) / "m2_done.json", dest / f"{name}_m2_done.json")
if SAVE_MERGED_TABLES:
    shutil.copytree(MERGED_TABLES, dest / "tables", dirs_exist_ok=True)
else:
    shutil.copy2(MERGED_TABLES / "tables.json", dest / "tables.json")
print("Saved:", dest)
print("Do not push these files or the raw corpus to the GitHub source repository.")